In [1]:
from typing import Tuple

from ibm_noise_models import *
from qiskit.quantum_info import Kraus
from qiskit_aer.noise import QuantumError


def ibm_get_minimal_cptp(error: QuantumError) -> Kraus:
    channel = error.to_quantumchannel()
    return Kraus(channel)

def get_ibm_op_str(instruction: Instruction) -> str:
    assert isinstance(instruction, Instruction)
    return instruction.op.value.lower()

def my_get_quantum_channel(error: QuantumError, only_kraus=False) -> List[Tuple[List[Union[str, KrausOperator]], float]]:
    result = []
    for prob, circ in zip(error.probabilities, error.circuits):
        list_error = []
        for ci in circ.data:
            inst = ci.operation
            if inst.name == "kraus":
                list_error.append(KrausOperator(inst.params, 0, factorize=True))
            elif not only_kraus:
                list_error.append(inst)

        result.append((list_error, prob))
    return result


def get_all_kraus_operators(hardware: HardwareSpec, op_: Optional[str] = None, q0: int = None, q1: int= None):
    result = []
    noise_model = get_ibm_noise_model(hardware, thermal_relaxation=True)
    quantum_errors = noise_model._local_quantum_errors

    for (op, op_dict) in quantum_errors.items():
        if op is None or op == op_:
            for (q_tuple, error) in op_dict.items():
                if (q0 is None or q0 == q_tuple[0]) and (q1 is None or q1 == q_tuple[1]):
                    temp_result = my_get_quantum_channel(error, True)
                    for tt in temp_result:
                        for t in tt[0]:
                            result.append(t)

    return result

In [12]:
noise_model = get_ibm_noise_model(HardwareSpec.ALGIERS, thermal_relaxation=True)
error = noise_model._local_quantum_errors['x'][(0,)]
KrausOperator(Kraus(error.circuits[0]).data, 0, factorize=True).simplified_ops

['I']

In [13]:
for prob, circ in zip(error.probabilities, error.circuits):
    print(circ)
    print(len(Kraus(circ).data), prob, Kraus(circ))
    print()

   ┌───┐┌───┐
q: ┤ I ├┤ I ├
   └───┘└───┘
1 0.9996000320208661 Kraus([[[-1.+0.j,  0.+0.j],
        [ 0.+0.j, -1.+0.j]]],
      input_dims=(2,), output_dims=(2,))

   ┌───┐┌───┐
q: ┤ I ├┤ Z ├
   └───┘└───┘
1 4.910521220967494e-06 Kraus([[[-1.+1.2246468e-16j,  0.+0.0000000e+00j],
        [ 0.+0.0000000e+00j,  1.+0.0000000e+00j]]],
      input_dims=(2,), output_dims=(2,))

   ┌───┐     
q: ┤ I ├─|0>─
   └───┘     
2 0.0002823753383841272 Kraus([[[1.+0.j, 0.+0.j],
        [0.+0.j, 0.+0.j]],

       [[0.+0.j, 1.+0.j],
        [0.+0.j, 0.+0.j]]],
      input_dims=(2,), output_dims=(2,))

   ┌───┐┌───┐
q: ┤ X ├┤ I ├
   └───┘└───┘
1 3.7549914633661454e-05 Kraus([[[ 0.+0.j, -1.+0.j],
        [-1.+0.j,  0.+0.j]]],
      input_dims=(2,), output_dims=(2,))

   ┌───┐┌───┐
q: ┤ X ├┤ Z ├
   └───┘└───┘
1 1.8446343212028164e-10 Kraus([[[ 0.+0.0000000e+00j, -1.+1.2246468e-16j],
        [ 1.+0.0000000e+00j,  0.+0.0000000e+00j]]],
      input_dims=(2,), output_dims=(2,))

   ┌───┐     
q: ┤ X ├─|0>─
   └─

In [7]:
for prob, circ in zip(error.probabilities, error.circuits):
        list_error = []
        print(circ)


   ┌───┐┌───┐
q: ┤ I ├┤ I ├
   └───┘└───┘
   ┌───┐┌───┐
q: ┤ I ├┤ Z ├
   └───┘└───┘
   ┌───┐     
q: ┤ I ├─|0>─
   └───┘     
   ┌───┐┌───┐
q: ┤ X ├┤ I ├
   └───┘└───┘
   ┌───┐┌───┐
q: ┤ X ├┤ Z ├
   └───┘└───┘
   ┌───┐     
q: ┤ X ├─|0>─
   └───┘     
   ┌───┐┌───┐
q: ┤ Y ├┤ I ├
   └───┘└───┘
   ┌───┐┌───┐
q: ┤ Y ├┤ Z ├
   └───┘└───┘
   ┌───┐     
q: ┤ Y ├─|0>─
   └───┘     
   ┌───┐┌───┐
q: ┤ Z ├┤ I ├
   └───┘└───┘
   ┌───┐┌───┐
q: ┤ Z ├┤ Z ├
   └───┘└───┘
   ┌───┐     
q: ┤ Z ├─|0>─
   └───┘     


In [10]:
print(len(error.circuits))

12
